# Clasificación bancaria con una red neuronal

[Abrir y ejecutar en Google Colab](https://colab.research.google.com/github/al073016/Practica-4-Modelos-de-machine-learning/blob/main/01_Clasificacion_Bancaria.ipynb)

Pregunta: ¿qué clientes podrían contratar un depósito a plazo? Fuente: [Bank Marketing, UCI](https://archive.ics.uci.edu/dataset/222/bank+marketing), licencia CC BY 4.0. Se usa `bank-full.csv` (45 211 filas). Se excluye `duration`: antes de llamar al cliente no se conoce cuánto durará la llamada. La selección del modelo y del umbral se hace solo con validación; prueba se reserva para la evaluación final.

**Uso:** abre este notebook en Google Colab y elige **Entorno de ejecución → Ejecutar todo**. Las celdas descargan los datos automáticamente. La última celda permite probar una muestra del conjunto de prueba. El entrenamiento en CPU puede tardar varios minutos.

## 1. Bibliotecas y descarga automática

In [ ]:
import importlib.util, subprocess, sys
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
          'sklearn': 'scikit-learn', 'tensorflow': 'tensorflow', 'openpyxl': 'openpyxl'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
import io, zipfile, urllib.request
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report, RocCurveDisplay, PrecisionRecallDisplay

url = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
blob = urllib.request.urlopen(url, timeout=60).read()
with zipfile.ZipFile(io.BytesIO(blob)) as outer:
    nested = outer.read('bank.zip')
with zipfile.ZipFile(io.BytesIO(nested)) as inner:
    df = pd.read_csv(io.BytesIO(inner.read('bank-full.csv')), sep=';')
assert len(df) == 45211 and 'y' in df
print(df.shape, '\nObjetivo:', df['y'].value_counts(normalize=True), '\nFaltantes:', df.isna().sum().sum())
display(df.head())

## 2. División y preparación sin fuga de información
`unknown` se conserva como categoría. El codificador y el escalador se ajustan únicamente con entrenamiento.

In [ ]:
X = df.drop(columns=['y', 'duration'])
y = (df['y'] == 'yes').astype(int).to_numpy()
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)
numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
prep = ColumnTransformer([('num', StandardScaler(), numeric), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical)])
X_train_p = prep.fit_transform(X_train).astype('float32')
X_val_p = prep.transform(X_val).astype('float32')
X_test_p = prep.transform(X_test).astype('float32')
print('Tamaños:', len(X_train), len(X_val), len(X_test), 'Variables transformadas:', X_train_p.shape[1])

## 3. Ajuste de arquitectura
Comparamos dos tamaños de MLP en validación. `EarlyStopping` y `ReduceLROnPlateau` controlan el entrenamiento. Se usa peso de clase para atender la minoría positiva.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
class_weight = {0: float(weights[0]), 1: float(weights[1])}

def build_model(widths):
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(X_train_p.shape[1],))])
    for width in widths:
        model.add(tf.keras.layers.Dense(width, activation='relu'))
        model.add(tf.keras.layers.Dropout(0.15))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

trials = []
for widths in [(32, 16), (64, 32)]:
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(widths)
    history = model.fit(X_train_p, y_train, validation_data=(X_val_p, y_val),
                        epochs=50, batch_size=128, class_weight=class_weight, verbose=0,
                        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
                                   tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5)])
    p_val = model.predict(X_val_p, verbose=0).ravel()
    auc = roc_auc_score(y_val, p_val)
    trials.append((auc, widths, model, history, p_val))
    print(widths, 'épocas:', len(history.history['loss']), 'ROC-AUC validación:', round(auc, 4))

best_auc, best_widths, model, history, p_val = max(trials, key=lambda t: t[0])
thresholds = np.arange(0.05, 0.951, 0.01)
f1s = [f1_score(y_val, p_val >= t) for t in thresholds]
threshold = float(thresholds[np.argmax(f1s)])
print('Modelo elegido:', best_widths, 'umbral:', round(threshold, 2), 'F1 validación:', round(max(f1s), 4))

## 4. Evaluación final en prueba
La exactitud sola puede ocultar errores en la clase minoritaria. Se reportan ROC-AUC, PR-AUC, F1, precisión y recall.

In [ ]:
p_test = model.predict(X_test_p, verbose=0).ravel()
y_pred = (p_test >= threshold).astype(int)
print('ROC-AUC:', round(roc_auc_score(y_test, p_test), 4))
print('PR-AUC:', round(average_precision_score(y_test, p_test), 4))
print('Referencia PR-AUC por prevalencia:', round(float(y_test.mean()), 4))
print(classification_report(y_test, y_pred, target_names=['No contrata','Contrata']))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['No','Sí'], ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test, p_test, ax=axes[2])
plt.tight_layout(); plt.show()
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida'); plt.legend(); plt.show()

## 5. Probar una predicción
Cambia `indice` por cualquier posición válida del conjunto de prueba. La predicción es una probabilidad, no una decisión financiera automática.

In [ ]:
indice = 0
assert 0 <= indice < len(X_test)
display(X_test.iloc[[indice]])
print('Probabilidad de contratar:', round(float(p_test[indice]), 3))
print('Predicción:', 'Sí' if y_pred[indice] else 'No', '| Real:', 'Sí' if y_test[indice] else 'No')